# v2 · Fase 3 — Exportación del modelo a ONNX (`imgsz=416`)

Convertimos `model/v2/best.pt` a ONNX con `imgsz=416` (mismo tamaño que el entrenamiento) para inferencia móvil con `onnxruntime-web`.

**Configuración:**
- `imgsz=416` (consistente con training).
- `opset=12` (compatible con onnxruntime-web).
- `simplify=True` (optimiza el grafo).
- `dynamic=False` (batch fijo = 1).

Esperado: input `(1, 3, 416, 416)` float32, output `(1, 9, 3549)` — donde 3549 = 52² + 26² + 13² (anchors a strides 8/16/32 sobre 416).

In [1]:
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
from pathlib import Path
import shutil
from ultralytics import YOLO

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "dataset_v2").exists():
    ROOT = ROOT.parent
MODEL_DIR = ROOT / "model" / "v2"
BEST_PT = MODEL_DIR / "best.pt"
assert BEST_PT.exists(), f"No se encontró {BEST_PT}"

model = YOLO(str(BEST_PT))
onnx_path = model.export(
    format="onnx",
    imgsz=416,
    opset=12,
    simplify=True,
    dynamic=False,
    half=False,
    device="cpu",
)
print(f"Exportado: {onnx_path}")

Ultralytics 8.4.47 🚀 Python-3.12.7 torch-2.11.0 CPU (Apple M3 Pro)


Model summary (fused): 73 layers, 3,006,623 parameters, 0 gradients, 8.1 GFLOPs



PyTorch: starting from '/Users/ccepeda/Documents/Personales/Maestria/SistemaRecomendacion/model/v2/best.pt' with input shape (1, 3, 416, 416) BCHW and output shape(s) (1, 9, 3549) (5.9 MB)



ONNX: starting export with onnx 1.21.0 opset 12...


ONNX: slimming with onnxslim 0.1.92...


ONNX: export success ✅ 0.6s, saved as '/Users/ccepeda/Documents/Personales/Maestria/SistemaRecomendacion/model/v2/best.onnx' (11.6 MB)



Export complete (0.6s)
Results saved to /Users/ccepeda/Documents/Personales/Maestria/SistemaRecomendacion/model/v2/best.onnx
Predict:         yolo predict task=detect model=/Users/ccepeda/Documents/Personales/Maestria/SistemaRecomendacion/model/v2/best.onnx imgsz=416 
Validate:        yolo val task=detect model=/Users/ccepeda/Documents/Personales/Maestria/SistemaRecomendacion/model/v2/best.onnx imgsz=416 data=/Users/ccepeda/Documents/Personales/Maestria/SistemaRecomendacion/dataset_v2/data_train.yaml  
Visualize:       https://netron.app


Exportado: /Users/ccepeda/Documents/Personales/Maestria/SistemaRecomendacion/model/v2/best.onnx


In [2]:
# Copia: la versión definitiva queda en model/v2/best.onnx y
# en app/public/models/coin-detector.onnx para servir desde la app.
src = Path(onnx_path)
model_dst = MODEL_DIR / "best.onnx"
app_models = ROOT / "app" / "public" / "models"
app_models.mkdir(parents=True, exist_ok=True)
app_dst = app_models / "coin-detector.onnx"

if src.resolve() != model_dst.resolve():
    shutil.copy2(src, model_dst)
shutil.copy2(src, app_dst)
for p in [model_dst, app_dst]:
    print(f"  {p} ({p.stat().st_size / 1024 / 1024:.2f} MB)")

  /Users/ccepeda/Documents/Personales/Maestria/SistemaRecomendacion/model/v2/best.onnx (11.61 MB)
  /Users/ccepeda/Documents/Personales/Maestria/SistemaRecomendacion/app/public/models/coin-detector.onnx (11.61 MB)


## Verificación: input/output shapes y latencia local

In [3]:
import onnxruntime as ort
import numpy as np
import time

session = ort.InferenceSession(str(model_dst), providers=["CPUExecutionProvider"])
for inp in session.get_inputs():
    print(f"INPUT  {inp.name}: shape={inp.shape}, type={inp.type}")
for out in session.get_outputs():
    print(f"OUTPUT {out.name}: shape={out.shape}, type={out.type}")

# Benchmark de latencia (warm-up 3 veces, luego mide 20)
dummy = np.random.rand(1, 3, 416, 416).astype(np.float32)
name = session.get_inputs()[0].name
for _ in range(3):
    session.run(None, {name: dummy})
t0 = time.perf_counter()
N = 20
for _ in range(N):
    session.run(None, {name: dummy})
elapsed = time.perf_counter() - t0
print(f"\nLatencia local CPU (imgsz=416): {elapsed/N*1000:.1f} ms/inferencia ({N/elapsed:.1f} FPS)")
print("En móvil con onnxruntime-web (WASM single-thread) suele ser 2-4× más lento que CPU desktop.")

INPUT  images: shape=[1, 3, 416, 416], type=tensor(float)
OUTPUT output0: shape=[1, 9, 3549], type=tensor(float)



Latencia local CPU (imgsz=416): 11.2 ms/inferencia (89.5 FPS)
En móvil con onnxruntime-web (WASM single-thread) suele ser 2-4× más lento que CPU desktop.


## Próximos pasos

1. Actualizar `app/src/lib/yolo.ts`: cambiar `IMG_SIZE = 640` a `IMG_SIZE = 416`.
2. Reconstruir la app y validar latencia real en el navegador / WebView.
3. Si la latencia móvil sigue insuficiente, considerar:
   - Quantización INT8 (Ultralytics: `model.export(format='onnx', int8=True)`).
   - Reducir aún más a `imgsz=320` (con re-entrenamiento a ese tamaño).
   - Usar `half=True` (FP16) si el navegador lo soporta.